In [ ]:
import os
from pathlib import Path

import torch

os.environ.setdefault("TRITON_CACHE_DIR", str(Path(".triton-cache").resolve()))

import triton
import triton.language as tl

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}") # type:ignore
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:

# 1. THE KERNEL (This code executes directly on the GPU cores)
@triton.qq
def add_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    # Figure out which parallel instance (program) this execution thread belongs to
    pid = tl.program_id(axis=0)
    
    # Calculate the memory offset boundaries for this specific block
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    
    # Create a boundary guard mask so we don't read out-of-bounds memory
    mask = offsets < n_elements
    
    # Load data from slow Global Memory (DRAM) into ultra-fast on-chip registers
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    
    # Perform the execution in parallel
    output = x + y
    
    # Write the results back out to Global Memory
    tl.store(output_ptr + offsets, output, mask=mask)

# 2. THE HOST WRAPPER (This coordinates execution from the CPU)
def triton_add(x: torch.Tensor, y: torch.Tensor):
    output = torch.empty_like(x)
    n_elements = output.numel()
    
    # Define the Grid: how many parallel blocks do we need to launch?
    # cdiv divides and rounds up (e.g., if elements don't perfectly fit into blocks)
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    
    # Launch the kernel on the GPU
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output

# 3. VERIFICATION CODE
size = 100_000
x = torch.rand(size, device='cuda')
y = torch.rand(size, device='cuda')

# Run native PyTorch operation
output_torch = x + y

# Run your custom GPU kernel
output_triton = triton_add(x, y)

# Compare the results
max_diff = torch.max(torch.abs(output_torch - output_triton)).item()
print(f"Kernel Execution Successful!")
print(f"Max numerical difference from PyTorch: {max_diff}")

In [ ]:
# No extra compiler setup needed for the Triton benchmark.

In [7]:
!pip install triton-windows

   ---------------------------------------- 0.0/49.6 MB ? eta -:--:--
   - -------------------------------------- 1.3/49.6 MB 8.4 MB/s eta 0:00:06
   -- ------------------------------------- 3.1/49.6 MB 7.7 MB/s eta 0:00:07
   --- ------------------------------------ 4.7/49.6 MB 7.7 MB/s eta 0:00:06
   ---- ----------------------------------- 6.0/49.6 MB 7.5 MB/s eta 0:00:06
   ------ --------------------------------- 7.6/49.6 MB 7.5 MB/s eta 0:00:06
   ------- -------------------------------- 9.2/49.6 MB 7.5 MB/s eta 0:00:06
   -------- ------------------------------- 10.7/49.6 MB 7.4 MB/s eta 0:00:06
   ---------- ----------------------------- 12.6/49.6 MB 7.6 MB/s eta 0:00:05
   ------------ --------------------------- 14.9/49.6 MB 8.1 MB/s eta 0:00:05
   -------------- ------------------------- 17.6/49.6 MB 8.5 MB/s eta 0:00:04
   ---------------- ----------------------- 20.2/49.6 MB 8.8 MB/s eta 0:00:04
   ------------------ --------------------- 22.8/49.6 MB 9.1 MB/s eta 0:00:03


In [8]:
import time
import os
from pathlib import Path

import torch

os.environ.setdefault("TRITON_CACHE_DIR", str(Path(".triton-cache").resolve()))

import triton
import triton.language as tl

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available on this machine/runtime.")

In [9]:
# CPU vs GPU benchmark using a Triton GPU kernel.
# This avoids the Windows cl.exe requirement from torch C++ extensions.

@triton.jit
def vector_add_kernel(a_ptr, b_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    a = tl.load(a_ptr + offsets, mask=mask)
    b = tl.load(b_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, a + b, mask=mask)

def triton_vector_add(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    out = torch.empty_like(a)
    n_elements = out.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)
    vector_add_kernel[grid](a, b, out, n_elements, BLOCK_SIZE=1024)
    return out

def time_cpu(fn, warmup=5, repeats=30):
    for _ in range(warmup):
        fn()

    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    end = time.perf_counter()
    return (end - start) * 1000 / repeats

def time_gpu(fn, warmup=10, repeats=100):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats

# Large enough to show GPU parallelism clearly.
n = 50_000_000

a_cpu = torch.rand(n, dtype=torch.float32)
b_cpu = torch.rand(n, dtype=torch.float32)

# Move data once before timing so the GPU timing measures computation, not PCIe transfer.
a_gpu = a_cpu.cuda()
b_gpu = b_cpu.cuda()

cpu_ms = time_cpu(lambda: a_cpu + b_cpu)
gpu_ms = time_gpu(lambda: triton_vector_add(a_gpu, b_gpu))

cpu_out = a_cpu + b_cpu
gpu_out = triton_vector_add(a_gpu, b_gpu)
max_diff = (cpu_out.cuda() - gpu_out).abs().max().item()

print(f"Elements: {n:,}")
print(f"CPU time: {cpu_ms:.3f} ms")
print(f"Triton GPU kernel time: {gpu_ms:.3f} ms")
print(f"Speedup: {cpu_ms / gpu_ms:.1f}x")
print(f"Max difference: {max_diff}")

Elements: 50,000,000
CPU time: 46.678 ms
Triton GPU kernel time: 3.374 ms
Speedup: 13.8x
Max difference: 0.0
